In [11]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(-1, '/Users/ponddie/Documents/py/mda_project/explo/yenha/lib')
from hahelper import *

pd.options.display.max_columns=100
pd.options.display.max_rows=1000

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:
df_raw = pd.read_csv(PATH_TRAFFIC + "data-2026-01.csv", header=None, names=TRAFFIC_COLS)
df_sites = pd.read_csv(FILEPATH_SITES, header=None, names=SITES_COLS)
df_rich = pd.read_csv(FILEPATH_RICH, header=None, names=RICH_COLS)

## 1. Feature Engineering

### 1.1 Traffic

In [13]:
df = df_raw.copy()
df = pd.read_csv(PATH_TRAFFIC + "data-2026-01.csv", header=None, names=DATA_COLS)
df = df.query('type=="FIETSERS"')
df["start_time"] = pd.to_datetime(df["start_time"])
df["date"] = df["start_time"].dt.strftime("%Y-%m-%d")
df["hour"] = df["start_time"].dt.hour
df["datetime"] = (pd.to_datetime(df["date"]) + pd.to_timedelta(df["hour"], unit="h"))
df_hourly = (
    df.groupby(["site_id", "datetime", "date", "hour", "direction"], as_index=False)
      .agg(count=("traffic", "sum"))
      .rename(columns={"count": "traffic"})
      .sort_values(["site_id", "datetime"]).reset_index(drop=True)
      )

In [14]:
df_fts_traffic_15min = compute_traffic_15min_features(df_raw)
df_fts_datetime = compute_datetime_features(df_hourly)
print(df_fts_traffic_15min.shape, df_fts_datetime.shape)
display(df_fts_traffic_15min.head(2)), display(df_fts_datetime.head(2))

(215760, 18) (215760, 22)


,site_id,datetime,date,hour,direction,traffic,traffic_00_15,traffic_15_30,traffic_30_45,traffic_45_60,traffic_15m_mean,traffic_15m_std,traffic_15m_min,traffic_15m_max,traffic_00_15_pct,traffic_15_30_pct,traffic_30_45_pct,traffic_45_60_pct
0,1,2026-01-01,2026-01-01,0,IN,0.0,0.0,0.0,0.0,0.0,0.00,0.0,0.0,0.0,NaN,NaN,NaN,NaN
1,1,2026-01-01,2026-01-01,0,OUT,3.0,0.0,0.0,3.0,0.0,0.75,1.5,0.0,3.0,0.0,0.0,1.0,0.0


,site_id,datetime,date,hour,direction,day,month,year,dow,woy,is_weekend,is_weekday,is_monday,is_friday,is_morning_rush,is_afternoon_rush,is_rush_hour,is_lunch_hour,is_night,is_business_hours,is_public_holiday,public_holiday
0,1,2026-01-01,2026-01-01,0,IN,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar
1,1,2026-01-01,2026-01-01,0,OUT,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar


(None, None)

### 1.2 Richtching - Metadata Site

In [15]:
df_fts_site = compute_site_features(df_hourly)
print(df_fts_site.shape)
df_fts_site.head(3)

(215760, 12)


,site_id,direction,date,hour,datetime,site_sensor_age,site_is_road_tunnel,site_is_road_national,site_is_road_ring,site_is_road_motorway,site_is_inbound,site_is_outbound
0,1,IN,2026-01-01,0,2026-01-01 00:00:00,2324,1,0,0,0,1,0
1,1,OUT,2026-01-01,0,2026-01-01 00:00:00,2324,1,0,0,0,0,1
2,1,IN,2026-01-01,1,2026-01-01 01:00:00,2324,1,0,0,0,1,0


### 1.3 Extra data: Population - Weather

In [16]:
df_fts_pop = pd.read_csv(FILEPATH_FTS_POP)
df_fts_weather = pd.read_csv(FILEPATH_FTS_WTHER)
df_fts_pop.shape, df_fts_weather.shape

((151, 8), (8806320, 18))

In [17]:
display(df_fts_pop.head(2))
display(df_fts_weather.head(2))

,site_id,siteno,longtitude,latitude,geometry,pop_1km,pop_5km,pop_10km
0,1,100046096,4.456122,50.916183,POINT (4.456121776137429 50.91618331151478),2754.790527,99128.390625,726663.7500
1,2,100052862,4.471690,51.275120,POINT (4.47169 51.27512),1369.083008,165217.234375,590641.4375


,site_id,longtitude,latitude,naam,region,datetime,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,wind_speed_10m,wind_direction_10m,pressure_msl,cloud_cover,date,hour,distance
0,1,4.456122,50.916183,Machelen,Leuven,2019-08-01T03:00,15.4,80,0.0,0.0,0.0,12.4,206,1017.3,84,2019-08-01,3,0.247072
1,1,4.456122,50.916183,Machelen,Leuven,2019-08-01T02:00,15.6,78,0.0,0.0,0.0,11.6,206,1017.6,84,2019-08-01,2,0.247072


In [18]:
df_fts_pop = df_fts_pop[['site_id', 'pop_1km', 'pop_5km', 'pop_10km']]
df_fts_weather = df_fts_weather[['site_id', 'date', 'hour', 'datetime', 'temperature_2m', 'relative_humidity_2m', 'precipitation', 'rain',
       'snowfall', 'wind_speed_10m', 'wind_direction_10m', 'pressure_msl', 'cloud_cover']]
df_fts_weather = df_fts_weather.rename(columns={x: f"wt_{x}" for x in df_fts_weather.columns if x not in ['site_id','datetime','date','hour']})
df_fts_weather["datetime"] = pd.to_datetime(df_fts_weather["datetime"])

In [36]:
df_fts_site.shape

(215760, 8)

In [20]:
df_fts_weather.head(2)

,site_id,date,hour,datetime,wt_temperature_2m,wt_relative_humidity_2m,wt_precipitation,wt_rain,wt_snowfall,wt_wind_speed_10m,wt_wind_direction_10m,wt_pressure_msl,wt_cloud_cover
0,1,2019-08-01,3,2019-08-01 03:00:00,15.4,80,0.0,0.0,0.0,12.4,206,1017.3,84
1,1,2019-08-01,2,2019-08-01 02:00:00,15.6,78,0.0,0.0,0.0,11.6,206,1017.6,84


### 1.4 Merge feature

In [21]:
meta_columns = ['site_id', 'direction', 'date', 'hour', 'datetime']
dfx = (
    df_hourly
    .merge(df_fts_datetime, on=meta_columns, how='left')
    .merge(df_fts_traffic_15min, on=meta_columns, how='left')
    .merge(df_fts_site, on=meta_columns, how='left')
    .merge(df_fts_pop, on='site_id', how='left')
    .merge(df_fts_weather, on=['site_id', 'date', 'hour', 'datetime'], how='left')
    )

In [22]:
dfx

,site_id,datetime,date,hour,direction,traffic_x,day,month,year,dow,woy,is_weekend,is_weekday,is_monday,is_friday,is_morning_rush,is_afternoon_rush,is_rush_hour,is_lunch_hour,is_night,is_business_hours,is_public_holiday,public_holiday,traffic_y,traffic_00_15,traffic_15_30,traffic_30_45,traffic_45_60,traffic_15m_mean,traffic_15m_std,traffic_15m_min,traffic_15m_max,traffic_00_15_pct,traffic_15_30_pct,traffic_30_45_pct,traffic_45_60_pct,site_sensor_age,site_is_road_tunnel,site_is_road_national,site_is_road_ring,site_is_road_motorway,site_is_inbound,site_is_outbound,pop_1km,pop_5km,pop_10km,wt_temperature_2m,wt_relative_humidity_2m,wt_precipitation,wt_rain,wt_snowfall,wt_wind_speed_10m,wt_wind_direction_10m,wt_pressure_msl,wt_cloud_cover
0,1,2026-01-01 00:00:00,2026-01-01,0,IN,0.0,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0,NaN,NaN,NaN,NaN,2324,1,0,0,0,1,0,2754.790527,99128.390625,726663.75000,0.4,97,0.0,0.0,0.0,16.2,240,1019.8,14
1,1,2026-01-01 00:00:00,2026-01-01,0,OUT,3.0,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar,3.0,0.0,0.0,3.0,0.0,0.75,1.50000,0.0,3.0,0.0,0.0,1.0,0.0,2324,1,0,0,0,0,1,2754.790527,99128.390625,726663.75000,0.4,97,0.0,0.0,0.0,16.2,240,1019.8,14
2,1,2026-01-01 01:00:00,2026-01-01,1,IN,1.0,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar,1.0,1.0,0.0,0.0,0.0,0.25,0.50000,0.0,1.0,1.0,0.0,0.0,0.0,2324,1,0,0,0,1,0,2754.790527,99128.390625,726663.75000,0.3,96,0.0,0.0,0.0,15.2,239,1019.1,35
3,1,2026-01-01 01:00:00,2026-01-01,1,OUT,0.0,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0,NaN,NaN,NaN,NaN,2324,1,0,0,0,0,1,2754.790527,99128.390625,726663.75000,0.3,96,0.0,0.0,0.0,15.2,239,1019.1,35
4,1,2026-01-01 02:00:00,2026-01-01,2,IN,2.0,1,1,2026,3,1,0,1,0,0,0,0,0,0,1,0,1,Nieuwjaar,2.0,1.0,0.0,0.0,1.0,0.50,0.57735,0.0,1.0,0.5,0.0,0.0,0.5,2324,1,0,0,0,1,0,2754.790527,99128.390625,726663.75000,0.6,95,0.0,0.0,0.0,16.5,239,1017.9,31
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
215755,152,2026-01-31 21:00:00,2026-01-31,21,OUT,0.0,31,1,2026,5,5,1,0,0,0,0,0,0,0,0,0,0,None,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0,NaN,NaN,NaN,NaN,92,0,1,0,0,0,1,1585.872559,47184.156250,239831.96875,6.2,88,0.0,0.0,0.0,7.6,185,1005.3,39
215756,152,2026-01-31 22:00:00,2026-01-31,22,IN,1.0,31,1,2026,5,5,1,0,0,0,0,0,0,0,1,0,0,None,1.0,0.0,1.0,0.0,0.0,0.25,0.50000,0.0,1.0,0.0,1.0,0.0,0.0,92,0,1,0,0,1,0,1585.872559,47184.156250,239831.96875,5.0,90,0.0,0.0,0.0,6.5,183,1005.6,5
215757,152,2026-01-31 22:00:00,2026-01-31,22,OUT,1.0,31,1,2026,5,5,1,0,0,0,0,0,0,0,1,0,0,None,1.0,1.0,0.0,0.0,0.0,0.25,0.50000,0.0,1.0,1.0,0.0,0.0,0.0,92,0,1,0,0,0,1,1585.872559,47184.156250,239831.96875,5.0,90,0.0,0.0,0.0,6.5,183,1005.6,5
215758,152,2026-01-31 23:00:00,2026-01-31,23,IN,0.0,31,1,2026,5,5,1,0,0,0,0,0,0,0,1,0,0,None,0.0,0.0,0.0,0.0,0.0,0.00,0.00000,0.0,0.0,NaN,NaN,NaN,NaN,92,0,1,0,0,1,0,1585.872559,47184.156250,239831.96875,4.6,91,0.0,0.0,0.0,7.0,177,1005.8,23


In [23]:
dfx.to_csv('data/data_training_v1_54col.csv',index=False)